# 06 - Explicabilidad de Modelos de IA (XAI)

Este cuaderno implementa técnicas de explicabilidad usando **SHAP** (SHapley Additive exPlanations) para interpretar modelos de detección de malware.

**Contenido:**
- Importancia de la explicabilidad en ciberseguridad
- Entrenamiento de un Random Forest sobre características PE
- Cálculo e interpretación de valores SHAP
- Gráfico de resumen y explicación individual

## 7.1 Importancia en ciberseguridad

Muchos modelos de IA, especialmente los de aprendizaje profundo, son considerados **cajas negras**. En ciberseguridad, la capacidad de explicar por qué un modelo tomó una decisión es fundamental para:

- **Auditorías de cumplimiento** (GDPR, NIS2)
- **Generación de evidencia digital forense**
- **Reducción de falsos positivos** validados por analistas

In [ ]:
# Instalar shap compatible con numpy < 2 (requerido por tensorflow)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'shap==0.44.1', '-q'])
print('[OK] shap 0.44.1 disponible')

## 7.2 SHAP: SHapley Additive exPlanations

In [ ]:
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# ---------------------------------------------------------------
# Generar dataset sintético si no existe (mismo que notebook 03)
# ---------------------------------------------------------------
if not os.path.exists('file_features.csv'):
    print('Generando dataset sintético de características PE...')
    rng = np.random.default_rng(42)
    n_benign, n_malicious = 800, 200
    benign = pd.DataFrame({
        'entry_point'        : rng.integers(4096, 8192, n_benign),
        'image_base'         : rng.integers(0x400000, 0x500000, n_benign),
        'size_of_image'      : rng.integers(50000, 200000, n_benign),
        'size_code_section'  : rng.integers(10000, 80000, n_benign),
        'dll_flag'           : rng.integers(0, 256, n_benign),
        'num_sections'       : rng.integers(3, 7, n_benign),
        'entropia_max'       : rng.uniform(4.0, 6.5, n_benign),
        'entropia_media'     : rng.uniform(3.0, 5.5, n_benign),
        'num_importaciones'  : rng.integers(50, 200, n_benign),
        'num_dlls_importadas': rng.integers(3, 10, n_benign),
        'num_exportaciones'  : rng.integers(0, 20, n_benign),
        'file_size'          : rng.integers(50000, 500000, n_benign),
        'label'              : 0
    })
    malicious = pd.DataFrame({
        'entry_point'        : rng.integers(4096, 8192, n_malicious),
        'image_base'         : rng.integers(0x400000, 0x500000, n_malicious),
        'size_of_image'      : rng.integers(50000, 200000, n_malicious),
        'size_code_section'  : rng.integers(10000, 80000, n_malicious),
        'dll_flag'           : rng.integers(0, 256, n_malicious),
        'num_sections'       : rng.integers(5, 12, n_malicious),
        'entropia_max'       : rng.uniform(6.5, 8.0, n_malicious),
        'entropia_media'     : rng.uniform(5.5, 7.5, n_malicious),
        'num_importaciones'  : rng.integers(200, 500, n_malicious),
        'num_dlls_importadas': rng.integers(8, 20, n_malicious),
        'num_exportaciones'  : rng.integers(0, 5, n_malicious),
        'file_size'          : rng.integers(100000, 1000000, n_malicious),
        'label'              : 1
    })
    df_features = pd.concat([benign, malicious], ignore_index=True)
    df_features.to_csv('file_features.csv', index=False)
    print(f'Dataset sintético guardado: {len(df_features)} muestras.')

# 1. Entrenar modelo
df = pd.read_csv('file_features.csv').dropna()
X, y = df.drop('label', axis=1), df['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
print(f'Modelo entrenado. Accuracy en test: {rf.score(X_test, y_test):.4f}')

In [ ]:
# 2. Calcular valores SHAP
explainer   = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# En shap 0.44.x con RandomForest, shap_values es una lista [clase_0, clase_1]
# Cada elemento tiene shape (n_muestras, n_features)
if isinstance(shap_values, list):
    sv_malicioso = shap_values[1]   # valores para clase 'malicioso'
    expected_val = explainer.expected_value[1]
else:
    # Versiones más nuevas devuelven un objeto Explanation
    sv_malicioso = shap_values.values[:, :, 1]
    expected_val = explainer.expected_value[1]

print(f'Forma de sv_malicioso: {sv_malicioso.shape}')
print('Valores SHAP calculados correctamente.')

In [ ]:
# 3. Gráfico de resumen (importancia global)
shap.summary_plot(
    sv_malicioso, X_test,
    plot_type='bar',
    show=False
)
plt.title('Importancia SHAP — Detección de Malware')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150)
plt.show()
print('Gráfico guardado: shap_summary.png')

In [ ]:
# 4. Gráfico de resumen tipo beeswarm (distribución de impacto)
shap.summary_plot(
    sv_malicioso, X_test,
    show=False
)
plt.title('Distribución de valores SHAP — Clase Malicioso')
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150)
plt.show()
print('Gráfico guardado: shap_beeswarm.png')

In [ ]:
# 5. Explicar una muestra individual (force plot)
# CORRECCIÓN: NO usar plt.tight_layout() con force_plot — distorsiona el gráfico
muestra_idx = 0

shap.force_plot(
    expected_val,
    sv_malicioso[muestra_idx, :],
    X_test.iloc[muestra_idx, :],
    matplotlib=True,
    show=False
)
plt.savefig('shap_force_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado: shap_force_plot.png')

# Mostrar valores SHAP de la muestra en tabla
muestra_shap = pd.Series(
    sv_malicioso[muestra_idx, :],
    index=X_test.columns
).sort_values(key=abs, ascending=False)

print(f'\nTop características para muestra {muestra_idx}:')
print(muestra_shap.head(10).to_string())

## Resumen

- Los valores SHAP muestran la contribución de cada característica a la predicción.
- Valores positivos empujan hacia la clase **malicioso**; negativos hacia **benigno**.
- El gráfico de resumen permite identificar qué características son más relevantes globalmente.
- El force plot explica una predicción individual, útil para auditorías forenses.